In [ ]:
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

import subprocess
import platform


# =========================
# 获取桌面路径
# =========================

desktop = os.path.join(os.path.expanduser("~"), "Desktop")
final_output_path = os.path.join(desktop, "B站专栏.xlsx")


# =========================
# 配置 Chrome（和视频代码一致）
# =========================

options = Options()

options.add_argument(
    r"--user-data-dir=C:\Users\12082\AppData\Local\Google\Chrome\SeleniumData"
)

options.add_argument("--start-maximized")


# 创建 Chrome WebDriver

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)


# =========================
# 打开B站专栏页面
# =========================

url = " " #输入B站专栏对应的网址

driver.get(url)


# =========================
# 等待页面加载
# =========================

wait = WebDriverWait(driver, 10)

try:

    wait.until(
        EC.presence_of_element_located(
            (By.CLASS_NAME, "list-content-item")
        )
    )

    print("页面加载完成")


except Exception as e:

    print("页面加载失败:", e)

    driver.quit()

    exit()


# =========================
# 提取专栏数据
# =========================

articles = driver.find_elements(
    By.CLASS_NAME,
    "list-content-item"
)


data = []


for article in articles:

    try:

        # 标题

        title = article.find_element(
            By.CLASS_NAME,
            "title"
        ).text.strip()


        # 浏览量

        view_count = article.find_element(
            By.CLASS_NAME,
            "view"
        ).text.strip()

        view_count = ''.join(
            filter(str.isdigit, view_count)
        )


        # 点赞数量

        like_count = article.find_element(
            By.CLASS_NAME,
            "like"
        ).text.strip()

        like_count = ''.join(
            filter(str.isdigit, like_count)
        )


        # 评论数量

        reply_count = article.find_element(
            By.CLASS_NAME,
            "reply"
        ).text.strip()

        reply_count = ''.join(
            filter(str.isdigit, reply_count)
        )


        data.append(
            {
                "标题": title,
                "浏览量": view_count,
                "评论数": reply_count,
                "点赞量": like_count
            }
        )


        print(
            f"标题:{title} | 浏览:{view_count} | 评论:{reply_count} | 点赞:{like_count}"
        )


    except Exception as e:

        print(
            "提取数据失败:",
            e
        )


# =========================
# 关闭浏览器
# =========================

driver.quit()



# =========================
# 数据处理
# =========================

df = pd.DataFrame(data)



if not df.empty:


    # 颠倒顺序

    df_reversed = (
        df.iloc[::-1]
        .reset_index(drop=True)
    )


    # 保留前300条

    df_filtered = df_reversed.head(300)



    # 保存Excel

    df_filtered.to_excel(
        final_output_path,
        index=False,
        sheet_name="文章数据"
    )



    # =========================
    # Excel格式调整
    # =========================

    wb = load_workbook(final_output_path)

    ws = wb.active



    # 第一列标题宽度

    ws.column_dimensions[
        get_column_letter(1)
    ].width = 813 / 7.5



    wb.save(final_output_path)



    print(
        f"处理后的数据已保存到: {final_output_path}"
    )



    # =========================
    # 自动打开Excel
    # =========================

    try:

        system_platform = platform.system()


        if system_platform == "Darwin":

            subprocess.run(
                ["open", final_output_path],
                check=True
            )


        elif system_platform == "Windows":

            os.startfile(
                final_output_path
            )


        elif system_platform == "Linux":

            subprocess.run(
                ["xdg-open", final_output_path],
                check=True
            )


        else:

            print(
                "无法识别操作系统"
            )


    except Exception as e:

        print(
            "无法打开Excel:",
            e
        )